# Wastewater surveillance, CDC NWSS (Lane B: the agent's output, captured)

**SISMID 2026 - Day 2, 9:00 - Data beyond Google Trends.**

> Captured example of what an agent produces from the Lane A prompts. Run as a backup or
> read as a reference.

Wastewater measures **biology (viral shedding)**, not behavior - independent of searching
or care-seeking, and it can **lead** clinical reporting. Source: CDC's National Wastewater
Surveillance System. Target: **Influenza A** in **Georgia** (ties to the capstone).


## Step 0: the helper (from the Step 0 prompt)

The CDC data is site-level and messy (many columns, string numerics). The helper pulls
Georgia flu-A samples and tidies date + concentration, with a cache fallback.


In [ ]:
# --- Produced by the agent from the Plan A / Step 0 prompt ---
import urllib.request, urllib.parse, json, os
import pandas as pd, matplotlib.pyplot as plt

# CDC NWSS 'Wastewater Data for Influenza A' (Socrata dataset ymmh-divb)
NWSS_ID = 'ymmh-divb'
COLS = ['sample_collect_date','site','counties_served','population_served',
        'pcr_target_avg_conc','pcr_target_flowpop_lin','pcr_target_detect']
CACHE_PATHS = ['../data/cdc_nwss_influenza_a_ga_cached.csv',
               'data/cdc_nwss_influenza_a_ga_cached.csv',
               './cdc_nwss_influenza_a_ga_cached.csv']

def nwss_fetch(state='ga'):
    """Influenza A wastewater samples for one state from CDC NWSS. DataFrame or None."""
    params = {'state_territory': state, '$select': ','.join(COLS),
              '$order': 'sample_collect_date', '$limit': '50000'}
    url = f'https://data.cdc.gov/resource/{NWSS_ID}.json?' + urllib.parse.urlencode(params)
    try:
        rows = json.loads(urllib.request.urlopen(url, timeout=45).read())
        return pd.DataFrame(rows)
    except Exception as e:
        print(f'CDC NWSS live pull failed ({type(e).__name__}): {e}')
        return None

def load_cache():
    for p in CACHE_PATHS:
        if os.path.exists(p):
            print(f'Using cached snapshot: {p}')
            return pd.read_csv(p)
    raise FileNotFoundError('NWSS cache not found; check the data/ folder.')

def get_ga_flu_wastewater():
    df = nwss_fetch('ga')
    if df is None or df.empty:
        df = load_cache()
    # tidy: parse date + numeric concentration
    df['date'] = pd.to_datetime(df['sample_collect_date'])
    df['conc'] = pd.to_numeric(df['pcr_target_avg_conc'], errors='coerce')
    return df


## Step 1: pull Georgia flu-A wastewater

Site-level samples across Georgia treatment plants, each with the counties and population
it serves.


In [ ]:
ww = get_ga_flu_wastewater()
print('rows:', len(ww), '| range:', ww['date'].min().date(), 'to', ww['date'].max().date())
print('distinct sites:', ww['site'].nunique())
print(ww[['date','site','counties_served','population_served','conc']].head())


## Step 2: aggregate to a state-level weekly signal

One site is noisy; a weekly mean across sites is the usable signal.


In [ ]:
wk = (ww.dropna(subset=['conc'])
        .set_index('date').groupby(pd.Grouper(freq='W'))['conc'].mean().reset_index())
plt.figure(figsize=(10,4))
plt.plot(wk['date'], wk['conc'])
plt.title('Georgia influenza A in wastewater (weekly mean concentration)')
plt.ylabel('copies/L'); plt.xlabel('date'); plt.tight_layout(); plt.show()


## Step 3: single site vs the aggregate

See why aggregation matters: pick the largest-population site and overlay it on the state
mean. One plant is spiky; the aggregate is smoother and more interpretable.


In [ ]:
ww['pop'] = pd.to_numeric(ww['population_served'], errors='coerce')
big = ww.sort_values('pop', ascending=False)['site'].iloc[0]
site = (ww[ww['site']==big].dropna(subset=['conc'])
          .set_index('date').groupby(pd.Grouper(freq='W'))['conc'].mean().reset_index())
plt.figure(figsize=(10,4))
plt.plot(wk['date'], wk['conc'], label='state weekly mean')
plt.plot(site['date'], site['conc'], alpha=0.6, label=f'site {big} only')
plt.legend(); plt.title('Single site vs state aggregate'); plt.ylabel('copies/L')
plt.xlabel('date'); plt.tight_layout(); plt.show()


## Step 4: sanity-check and save

Wastewater's gotchas: **site coverage changes over time**, single sites are noisy, and
there is a **reporting lag** (the last week or two is often incomplete).


In [ ]:
cov = ww.dropna(subset=['conc']).set_index('date').groupby(pd.Grouper(freq='W'))['site'].nunique()
print('reporting sites per week (last 6):'); print(cov.tail(6))
print('\nlatest sample date:', ww['date'].max().date(), '(expect a lag vs today)')
wk.to_csv('ga_flu_wastewater_weekly.csv', index=False)
print('saved ga_flu_wastewater_weekly.csv', wk.shape)


## Reflection

- Wastewater is a **different kind** of signal: biology, not behavior. No datacenter
  block, so retries/proxy rarely matter here - Google Trends was the hard case.
- On your own: overlay this weekly signal against **ILI** for Georgia and judge whether
  it **leads** (Santillana's wastewater early-warning preprint).
- **Stretch:** change the state, or the pathogen (the CDC catalog also has COVID, RSV-
  adjacent, measles, mpox datasets).
